# 01 — Validação dos talhões de teca

Este notebook faz uma conferência inicial dos arquivos vetoriais. **Não faça upload para um repositório público de arquivos que não estejam autorizados para divulgação.**

Os dados recebidos devem permanecer como cópia original; qualquer correção deve gerar uma nova versão documentada.


In [ ]:
!pip -q install geopandas matplotlib

In [ ]:
from google.colab import files
import os, zipfile, glob
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

uploads = files.upload()
pasta = '/content/talhoes_teca/originais'
os.makedirs(pasta, exist_ok=True)

for nome in uploads:
    if nome.lower().endswith('.zip'):
        with zipfile.ZipFile(nome) as z:
            z.extractall(pasta)

shps = glob.glob(os.path.join(pasta, '**', '*.shp'), recursive=True)
if not shps:
    raise FileNotFoundError('Nenhum .shp encontrado. Envie um ZIP com os arquivos auxiliares do shapefile.')
print('Shapefiles encontrados:')
for i, path in enumerate(shps): print(i, path)

# Se houver mais de um shapefile, ajuste o índice abaixo após conferir a lista.
gdf = gpd.read_file(shps[0])
print('Quantidade de feições:', len(gdf))
print('CRS:', gdf.crs)
print('Campos:', gdf.columns.tolist())
display(gdf.drop(columns='geometry').head(20))

In [ ]:
campos_esperados = ['CODTALHAO', 'CODPROJETO', 'ANO']
faltantes = [c for c in campos_esperados if c not in gdf.columns]
if faltantes:
    raise ValueError(f'Campos não encontrados: {faltantes}. Confira os nomes dos atributos.')
if gdf.crs is None:
    raise ValueError('CRS ausente: identifique o sistema de coordenadas no arquivo de origem antes de reprojetar.')

print('Geometrias inválidas:', int((~gdf.geometry.is_valid).sum()))
print('Geometrias vazias:', int(gdf.geometry.is_empty.sum()))
print('CRS original:', gdf.crs)

gdf_m = gdf.to_crs('EPSG:31981')
gdf_m['AREA_CALCULADA_HA'] = gdf_m.geometry.area / 10000
resumo = (gdf_m.groupby(['CODPROJETO','ANO'])
          .agg(numero_talhoes=('CODTALHAO','count'), area_calculada_ha=('AREA_CALCULADA_HA','sum'))
          .reset_index())
display(resumo)

fig, ax = plt.subplots(figsize=(10, 8))
gdf_m.plot(ax=ax, column='CODPROJETO', categorical=True, legend=True,
           edgecolor='black', linewidth=0.7, alpha=0.75)
for _, row in gdf_m.iterrows():
    p = row.geometry.representative_point()
    ax.annotate(f"{row['CODPROJETO']}-{row['CODTALHAO']}\\n{int(row['ANO'])}",
                (p.x, p.y), ha='center', va='center', fontsize=7)
ax.set_title('Talhões de teca por projeto e ano de plantio')
ax.set_xlabel('UTM (m)'); ax.set_ylabel('UTM (m)'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## Próxima etapa
Após validar os polígonos, registrar a data de execução e configurar o projeto Google Earth Engine, será criado o notebook de processamento Landsat e Sentinel-2. As coleções, datas, bandas e parâmetros usados serão exportados para metadados.